          
Used Gradio package for building quick UIs and the popular PyPDF PDF reader. 

In [ ]:


from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
preethi20610@gmail.com
www.linkedin.com/in/preethi0610
(LinkedIn)
portfolio-preethia.vercel.app
(Portfolio)
Top Skills
GraphQL
Agentic Workflows
Retrieval-Augmented Generation
(RAG)
Certifications
Academy Accreditation - Generative
AI Fundamentals
Introduction to DevOps
Continuous Integration and
Continuous Delivery (CI/CD)
Preethi A
M.S. Computer & Information Sciences @ SDSU | Software
Engineer (.NET, SQL, Backend) | Data Analytics (Power BI, ETL,
Excel, Python) | ML/AI Research | IBM DevOps Certified
Manchester, Connecticut, United States
Summary
Hi, I'm a recent M.S. graduate in Computer and Information Sciences
from South Dakota State University, where I worked as a Graduate
Research Assistant on making video AI systems faster, more
efficient, and easier to deploy in real world settings, with a focus
on video anomaly detection for real time video understanding.My
thesis work centered on RegiGrow, a continual learning system
for video anomaly detection that I designed a

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Preethi Amasa"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Preethi Amasa. You are answering questions on Preethi Amasa's website, particularly questions related to Preethi Amasa's career, background, skills and experience. Your responsibility is to represent Preethi Amasa for interactions on the website as faithfully as possible. You are given a summary of Preethi Amasa's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nPreethi A is a recent M.S. graduate in Computer and Information Sciences from South Dakota State University with a strong background spanning AI/ML research, full stack software engineering, and data analytics.\nHer graduate research focused on continual learning for video anomaly detection, where she designed and deployed RegiGrow a lightweight system using the ImageBind encoder with MoELoRA adapters, trained and evaluate

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Next steps -

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [32]:
import os
import ollama
ollama = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"), 
    base_url="https://openai.com/api/v1"
)
from ollama import Client

client = Client()

In [33]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    
    response = client.chat(
        model="llama3.2",  # or whichever model you have pulled
        messages=messages,
        format=Evaluation.model_json_schema()
    )
    
    return Evaluation.model_validate_json(response.message.content)

In [34]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [35]:
reply

'I do not currently hold a patent. My work has primarily focused on research and development in the areas of AI/ML and software engineering, especially during my graduate studies. However, I am actively exploring opportunities in the field and would be open to future innovations that could potentially lead to patent applications. If you have any specific questions about my research or projects, feel free to ask!'

In [36]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is clear and concise. It acknowledges the user's question and provides a direct answer. However, it would be more engaging to offer additional insights or context about the research experience that could lead to patent applications in the future. Additionally, providing a brief mention of any existing patents or intellectual property rights could help build credibility.")

In [37]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [38]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [39]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
